In [6]:
import sys
sys.path.append('../..')

from pathlib import Path
import os
from glob import glob

import cv2
from PIL import Image
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from src.model.baseline_cnn import LitCNN
from src.model.resnet_18 import LitResNet18
from src.model.dataset import ImageDataset

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
path_to_dataset = Path("../../data/01_raw/sneakers-dataset")

In [10]:
train_df_pre = pd.read_csv('../../data/01_raw/train_images.csv')

display(train_df_pre.head(), train_df_pre.shape)

,path,sneaker_class
0,nike_air_force_1_high/0036.jpg,nike_air_force_1_high
1,nike_air_jordan_1_high/0026.jpg,nike_air_jordan_1_high
2,converse_one_star/0025.jpg,converse_one_star
3,reebok_classic_leather/0004.jpg,reebok_classic_leather
4,nike_air_jordan_11/0014.jpg,nike_air_jordan_11


(4636, 2)

In [11]:
test_df = pd.read_csv('../../data/01_raw/test_images.csv')

display(test_df.head(), test_df.shape)

,path,sneaker_class
0,adidas_forum_low/0026.jpg,adidas_forum_low
1,nike_air_jordan_4/0078.jpg,nike_air_jordan_4
2,yeezy_boost_350_v2/0018.jpg,yeezy_boost_350_v2
3,adidas_superstar/0028.jpg,adidas_superstar
4,adidas_forum_high/0091.jpg,adidas_forum_high


(1160, 2)

In [12]:
train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.25,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

,path,sneaker_class
4439,converse_chuck_70_low/0086.jpg,converse_chuck_70_low
2029,nike_dunk_low/0038.jpg,nike_dunk_low
3641,vans_old_skool/0059.jpg,vans_old_skool
3041,adidas_stan_smith/0001.jpg,adidas_stan_smith
142,nike_air_jordan_1_high/0009.jpg,nike_air_jordan_1_high


(3477, 2)

,path,sneaker_class
455,new_balance_327/0028.jpg,new_balance_327
705,reebok_club_c_85/0070.jpg,reebok_club_c_85
3532,asics_gel-lyte_iii/0017.jpg,asics_gel-lyte_iii
4587,nike_dunk_high/0050.jpg,nike_dunk_high
1185,reebok_classic_leather/0019.jpg,reebok_classic_leather


(1159, 2)

,path,sneaker_class
0,adidas_forum_low/0026.jpg,adidas_forum_low
1,nike_air_jordan_4/0078.jpg,nike_air_jordan_4
2,yeezy_boost_350_v2/0018.jpg,yeezy_boost_350_v2
3,adidas_superstar/0028.jpg,adidas_superstar
4,adidas_forum_high/0091.jpg,adidas_forum_high


(1160, 2)

In [13]:
train_paths = train_df["path"].tolist()
val_paths   = val_df["path"].tolist()
test_paths  = test_df["path"].tolist()

train_labels = train_df["sneaker_class"].tolist()
val_labels   = val_df["sneaker_class"].tolist()
test_labels  = test_df["sneaker_class"].tolist()

In [15]:
train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

In [16]:
train_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=train_paths,
    labels=train_labels,
    augmenter=train_tfms
)

val_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=val_paths,
    labels=val_labels,
    augmenter=val_tfms
)

test_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=test_paths,
    labels=test_labels,
    augmenter=val_tfms
)

In [17]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

# Baseline CNN

In [19]:
model = LitCNN(num_classes=train_df_pre["sneaker_class"].nunique())

trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    logger=pl.loggers.TensorBoardLogger("lightning_logs", name="cnn_baseline"),
    log_every_n_steps=1
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


In [20]:
import warnings 
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

trainer.fit(model, train_loader, val_loader)


  | Name    | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | model   | Sequential       | 156 K  | train | 0    
1 | loss_fn | CrossEntropyLoss | 0      | train | 0    
-------------------------------------------------------------
156 K     Trainable params
0         Non-trainable params
156 K     Total params
0.628     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


In [22]:
pred_batches = trainer.predict(model, dataloaders=[test_loader])

Predicting: |          | 0/? [00:00<?, ?it/s]

In [23]:
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.24      0.23      0.24        30
           1       0.00      0.00      0.00        18
           2       0.11      0.17      0.13        30
           3       0.13      0.32      0.18        19
           4       0.00      0.00      0.00        14
           5       0.17      0.38      0.24        29
           6       0.19      0.32      0.24        19
           7       0.11      0.10      0.10        30
           8       0.10      0.11      0.10        18
           9       0.18      0.13      0.15        15
          10       0.27      0.30      0.29        30
          11       0.18      0.20      0.19        15
          12       0.22      0.11      0.15        18
          13       0.11      0.07      0.08        30
          14       0.00      0.00      0.00        22
          15       0.29      0.33      0.31        30
          16       0.15      0.10      0.12        30
          17       0.00    

# ResNet

In [24]:
resnet_model = LitResNet18(num_classes=train_df_pre["sneaker_class"].nunique())

resnet_trainer = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=pl.loggers.TensorBoardLogger("lightning_logs", name="resnet"),
    log_every_n_steps=1
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


In [25]:
import warnings 
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

resnet_trainer.fit(resnet_model, train_loader, val_loader)


  | Name    | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | model   | ResNet           | 11.2 M | train | 0    
1 | loss_fn | CrossEntropyLoss | 0      | train | 0    
-------------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.807    Total estimated model params size (MB)
69        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


In [26]:
resnet_pred_batches = resnet_trainer.predict(resnet_model, dataloaders=[test_loader])

Predicting: |          | 0/? [00:00<?, ?it/s]

In [27]:
y_pred_resnet = torch.cat(resnet_pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred_resnet))

              precision    recall  f1-score   support

           0       0.81      0.57      0.67        30
           1       0.91      0.56      0.69        18
           2       0.84      0.70      0.76        30
           3       1.00      0.53      0.69        19
           4       0.75      0.64      0.69        14
           5       0.76      0.76      0.76        29
           6       0.61      0.58      0.59        19
           7       0.84      0.70      0.76        30
           8       0.80      0.67      0.73        18
           9       0.50      0.60      0.55        15
          10       0.73      0.27      0.39        30
          11       0.78      0.47      0.58        15
          12       0.69      0.61      0.65        18
          13       0.88      0.77      0.82        30
          14       1.00      0.73      0.84        22
          15       0.69      0.80      0.74        30
          16       0.70      0.47      0.56        30
          17       0.28    